# Phase 4: Model Development

## 1. Project Setup

### Objective
The objective of this phase is to develop and compare traditional machine learning models for predicting household electricity consumption 10 minutes into the future.

### Approach
We will begin with a simple persistence baseline and gradually experiment with regression algorithms such as Linear Regression, Ridge, Lasso, Decision Tree, Random Forest, and Gradient Boosting.

### Evaluation
Models will be evaluated using MAE, RMSE, and MAPE on chronologically separated test data.

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
data_path = Path(r"D:\college ml project\EnergySense\data\processed\X_features.csv")
df_features = pd.read_csv(data_path)

data_output=Path(r"D:\college ml project\EnergySense\data\processed\y_target.csv")
df_output=pd.read_csv(data_output)
df_features

,timestamp,hour,day,month,weekends,lag_1,lag_10,lag_30,lag_60,lag_5,rolling_mean_5,rolling_mean_10,rolling_mean_30,rolling_std_10,Global_active_power
0,2006-12-16 18:24:00,18.0,5.0,12.0,1.0,2.926,2.388,2.720,4.216,2.852,3.0360,3.5828,3.892867,0.769995,3.452
1,2006-12-16 18:25:00,18.0,5.0,12.0,1.0,3.452,4.598,3.758,5.360,2.928,3.4244,3.6100,3.929933,0.813421,4.870
2,2006-12-16 18:26:00,18.0,5.0,12.0,1.0,4.870,4.524,4.342,5.374,2.940,3.8100,3.6444,3.947467,0.862182,4.868
3,2006-12-16 18:27:00,18.0,5.0,12.0,1.0,4.868,4.202,4.512,5.388,2.934,4.1964,3.7108,3.959267,0.932590,4.866
4,2006-12-16 18:28:00,18.0,5.0,12.0,1.0,4.866,4.472,4.058,3.666,2.926,4.2464,3.5812,3.929867,0.904688,3.176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2048265,2010-11-26 20:48:00,20.0,4.0,11.0,0.0,0.910,0.976,1.084,1.242,0.988,0.9368,0.9620,1.000400,0.036234,0.912
2048266,2010-11-26 20:49:00,20.0,4.0,11.0,0.0,0.912,0.984,1.086,1.218,0.982,0.9300,0.9584,0.995800,0.035588,0.948
2048267,2010-11-26 20:50:00,20.0,4.0,11.0,0.0,0.948,0.986,1.082,1.234,0.972,0.9752,0.9796,0.999667,0.084031,1.198
2048268,2010-11-26 20:51:00,20.0,4.0,11.0,0.0,1.198,0.990,1.086,1.400,0.908,0.9984,0.9830,0.997600,0.085178,1.024


In [12]:
df_features.shape

(2048270, 15)

In [13]:
df_output


,timestamp,target_10min
0,2006-12-16 18:24:00,3.538
1,2006-12-16 18:25:00,6.072
2,2006-12-16 18:26:00,4.536
3,2006-12-16 18:27:00,4.408
4,2006-12-16 18:28:00,2.912
...,...,...
2048265,2010-11-26 20:48:00,0.946
2048266,2010-11-26 20:49:00,0.944
2048267,2010-11-26 20:50:00,0.938
2048268,2010-11-26 20:51:00,0.934


In [16]:
df_features.index

RangeIndex(start=0, stop=2048270, step=1)

In [17]:
df_output.index

RangeIndex(start=0, stop=2048270, step=1)

In [18]:
df_features['timestamp'] = pd.to_datetime(
        df_features['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='raise'
    )
df_features = df_features.set_index('timestamp')
df_features.index

DatetimeIndex(['2006-12-16 18:24:00', '2006-12-16 18:25:00',
               '2006-12-16 18:26:00', '2006-12-16 18:27:00',
               '2006-12-16 18:28:00', '2006-12-16 18:29:00',
               '2006-12-16 18:30:00', '2006-12-16 18:31:00',
               '2006-12-16 18:32:00', '2006-12-16 18:33:00',
               ...
               '2010-11-26 20:43:00', '2010-11-26 20:44:00',
               '2010-11-26 20:45:00', '2010-11-26 20:46:00',
               '2010-11-26 20:47:00', '2010-11-26 20:48:00',
               '2010-11-26 20:49:00', '2010-11-26 20:50:00',
               '2010-11-26 20:51:00', '2010-11-26 20:52:00'],
              dtype='datetime64[ns]', name='timestamp', length=2048270, freq=None)

In [19]:
df_output['timestamp'] = pd.to_datetime(
        df_output['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='raise'
    )
df_output = df_output.set_index('timestamp')
df_output.index

DatetimeIndex(['2006-12-16 18:24:00', '2006-12-16 18:25:00',
               '2006-12-16 18:26:00', '2006-12-16 18:27:00',
               '2006-12-16 18:28:00', '2006-12-16 18:29:00',
               '2006-12-16 18:30:00', '2006-12-16 18:31:00',
               '2006-12-16 18:32:00', '2006-12-16 18:33:00',
               ...
               '2010-11-26 20:43:00', '2010-11-26 20:44:00',
               '2010-11-26 20:45:00', '2010-11-26 20:46:00',
               '2010-11-26 20:47:00', '2010-11-26 20:48:00',
               '2010-11-26 20:49:00', '2010-11-26 20:50:00',
               '2010-11-26 20:51:00', '2010-11-26 20:52:00'],
              dtype='datetime64[ns]', name='timestamp', length=2048270, freq=None)

In [20]:
df_features.index.equals(df_output.index)

True

In [22]:
df_features.shape

(2048270, 14)